# totalVI model training on longitudinal CITE-seq data

This notebook loads the preprocessed MuData object generated in the first notebook and trains a totalVI model to obtain an integrated RNA + ADT protein latent representation.

## 1. Import libraries

In [ ]:
import os

import numpy as np
import pandas as pd
import scipy

import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import muon
import scvi
import torch

## 2. Initial configuration

In [ ]:
# Check modules
print("scanpy:", sc.__version__)
print("scvi-tools:", scvi.__version__)
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

# Set seed for reproducibility
scvi.settings.seed = 0

# Figure configuration
sc.set_figure_params(figsize=(6, 6), frameon=False)
sns.set_theme()

%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"

# PyTorch configuration
torch.set_float32_matmul_precision("high")


## 3. Load preprocessed MuData object

In [ ]:
mdata = muon.read_h5mu("../data/processed/citeseq_longitudinal_preprocessed.h5mu")

mdata

In [ ]:
print("RNA:", mdata.mod["rna"].shape)
print("RNA subset:", mdata.mod["rna_subset"].shape)
print("Protein:", mdata.mod["prot"].shape)
print("RNA subset layers:", mdata.mod["rna_subset"].layers.keys())

## 4. Prepare MuData object for totalVI

In [ ]:
# Convert sparse matrices to dense arrays
mdata.mod["prot"].X = mdata.mod["prot"].X.toarray()
mdata.mod["rna_subset"].X = mdata.mod["rna_subset"].X.toarray()
mdata.mod["rna_subset"].layers["counts"] = mdata.mod["rna_subset"].layers["counts"].toarray()


# Register the MuData object for totalVI
scvi.model.TOTALVI.setup_mudata(
    mdata,
    rna_layer="counts",
    protein_layer=None,
    batch_key="batch",
    modalities={
        "rna_layer": "rna_subset",
        "protein_layer": "prot",
        "batch_key": "rna_subset",
    },
)

mdata

## 5. Train totalVI model

In [ ]:
model = scvi.model.TOTALVI(mdata)

model.train(early_stopping=True)

## 6. Check model training history

In [ ]:
last_val_valid = np.array(model.history["elbo_validation"])[-1]
last_val_train = np.array(model.history["elbo_train"])[-1]

global_min_loss = min(
    np.min(model.history["elbo_train"]),
    np.min(model.history["elbo_validation"]),
)

last_max_loss = max(last_val_train, last_val_valid)[0]

global_max_loss = max(
    np.max(model.history["elbo_train"]),
    np.max(model.history["elbo_validation"]),
)

In [ ]:
# Compute the min and max of both train and validation losses
min_loss = min(min(last_val_train, last_val_valid), global_min_loss)
max_loss = max(max(last_val_train, last_val_valid), global_max_loss)

ylim_min = 0.995 * min_loss  # 0.5% below the minimum
ylim_max = min(
    global_max_loss,
    ylim_min + (last_max_loss - ylim_min) * 4,
)  # keep it under the 25% part of figure

In [ ]:
fig, ax = plt.subplots(1, 1)

model.history["elbo_train"].plot(ax=ax, label="train")
model.history["elbo_validation"].plot(ax=ax, label="validation")

if isinstance(ylim_min, (int, float)) and isinstance(ylim_max, (int, float)):
    ax.set(title="Negative ELBO over training epochs", ylim=(ylim_min, ylim_max))
else:
    ax.set(title="Negative ELBO over training epochs")

ax.legend()

In [ ]:
os.makedirs("../results/figures", exist_ok=True)

fig.savefig(
    "../results/figures/totalvi_training_elbo.png",
    dpi=300,
    bbox_inches="tight"
)

## 7. Save trained model and latent representation

In [ ]:
# Store totalVI latent representation in the MuData object
mdata.mod["rna_subset"].obsm["X_totalVI"] = model.get_latent_representation()

# Define output paths
trained_mdata_path = "../data/processed/citeseq_longitudinal_totalvi_trained.h5mu"
model_dir = "../results/models/totalvi_longitudinal_model"

# Create model directory if it does not exist
os.makedirs(model_dir, exist_ok=True)

# Save trained MuData object and totalVI model
mdata.write(trained_mdata_path)
model.save(model_dir, overwrite=True)

print(f"Trained MuData object saved to: {trained_mdata_path}")
print(f"totalVI model saved to: {model_dir}")

In [ ]:
!ls -lh ../data/processed/
!ls -lh ../results/models/